# **TP3 | Introduction au chiffrement par blocs**

À la fin de ce TP on saura :
1. chiffrer & déchiffrer des données avec AES en mode ECB (Electronic Code Book) en utilisant le module `cryptography`
2. utiliser le padding PKCS#7
3. lire et écrire des données en [Base64](https://fr.wikipedia.org/wiki/Base64)

## 3.1. Chiffrement AES avec `cryptography`

Chiffrement par blocs de 128 bits (16 octets), avec une clé de 128, 192 ou 256 bits (16, 24 ou 32 octets). L'algorithme est particulièrement complexe, mais il faut quand même apprendre à bien l'utiliser : en 2013, une mauvaise utilisation de chiffrement par bloc a mené à une gigantesque [fuite de mots de passe](https://github.com/thaliaarchi/adobe-cred) chez Adobe !

Le nombre de bits de sécurité de ce cryptosystème correspond à la taille de sa clé. Dans ce cas [pourquoi](https://www.reddit.com/r/AskNetsec/comments/beisil/is_aes256_indeed_more_secure_than_aes128_or_is/) aller jusqu'à 256 bits ? Peut-être la peur des ordinateurs quantiques avec lesquels le nombre de bits de sécurité seront divisés par 2.

**TODO**  
Ci-dessous on définit un cryptosystème AES avec la clé `key`. Quel est le niveau de sécurité du système ?

In [2]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

key = b"ALL THEM WITCHES"
cipher = Cipher(algorithms.AES(key), modes.ECB())

### 3.1.a. Le chiffrement

La classe `Cipher` de `cryptography` a une méthode `.encryptor()` qui renvoie un objet `CipherContext`.

Cet objet fonctionne comme un buffer qu'on met à jour avec `.update` et qu'on *flush* avec `.finalize()`. Ces méthodes renvoient des `bytes` qu'il faut concaténer avec `+`.

**TODO**  
Chiffrer les données texte
```txt
données de test
```

In [3]:
data = "données de test".encode()
encrypt = cipher.encryptor()
cipher_data = encrypt.update(data)
cipher_data += encrypt.finalize()
print("Données chiffrées :", cipher_data)

Données chiffrées : b'\x82\x1eXm_\x16}SeF\xcb\x82\xd4\xe9\x16\x15'


### 3.1.b. Déchiffrement

Le déchiffrement fonctionne de la même manière, mais est créé avec `.decryptor()`.

**TODO**  
Vérifier que le message chiffré précédemment se déchiffre bien.

In [5]:
decrypt = cipher.decryptor()
#deciph_data = 

### 3.1.c. Chiffrement de texte « quelconque »

Contrairement au chiffrement "flux" qu'on a vu aux TP précédents où les données pouvaient être de taille quelconque, AES est un chiffrement « par blocs » pour lequel le message doit être décomposé en blocs de 16 octets.

**TODO**  
Essayer de chiffrer le texte ci-dessous.
```txt
Ces données ne font pas un multiple de 16 octets.
```
Que se passe-t-il ?

In [6]:
data = "Ces données ne font pas un multiple de 16 octets.".encode()
encrypt = cipher.encryptor()
cipher_data = encrypt.update(data)
try:
    cipher_data += encrypt.finalize() # ferme le flux
except ValueError as e:
    print("ValueError:", e)
    
print("Taille des données en clair :", len(data))
print("Taille des données chiffrées :", len(cipher_data))

ValueError: The length of the provided data is not a multiple of the block length.
Taille des données en clair : 50
Taille des données chiffrées : 48


## 3.2. Padding PKCS#7

Pour assurer que la taille de bloc soit respectée, on applique un **padding**, c'est-à-dire qu'on rajoute des octets à la fin des données pour atteindre un multiple de la taille de bloc.

Le méthode de padding la plus répandue est le [padding PKCS#7](https://datatracker.ietf.org/doc/html/rfc5652#section-6.3). Pour une taille de bloc `B`, elle consiste à rajouter `n` octets de valeur `n` à la fin pour que la taille des données devienne un multiple de `B`. Si la taille des données est déjà un multiple de la taille de bloc, on rajoute un bloc de `B` octets qui ont tous la valeur `B`.

Par [exemple](https://node-security.com/posts/cryptography-pkcs-7-padding/) sur des **blocs de 8 octets**, on aurait les paddings suivants :

Exemple 1  
![](padding_example1.svg)

Exemple 2  
![](padding_example2.svg)

Exemple 3  
![](padding_example_edge.svg)

**TODO (théorie)**  
Quelle est la taille de bloc maximale autorisée pour padding PKCS#7 ?

> 255 octets ! C'est beaucoup.

### 3.2.a. Fonction de padding

Le module `cryptography` implémente le padding PKCS#7 avec une syntaxe identique à celle du chiffrement, avec `.update` et `.finalize`.

**TODO**  
Utiliser le padding sur les exemples décrits précédemment.

In [7]:
from cryptography.hazmat.primitives import padding

test1 = bytes.fromhex("3a5e29")
test2 = bytes.fromhex("3a5e29f09f9094f09fa59ae29d93")
test3 = bytes.fromhex("3a5e29f09f9094f0")

pad = padding.PKCS7(8 * 8)

padder = pad.padder()
padded1 = padder.update(test1) + padder.finalize()
print(padded1.hex())
padder = pad.padder()
padded2 = padder.update(test2) + padder.finalize()
print(padded2.hex())
padder = pad.padder()
padded3 = padder.update(test3) + padder.finalize()
print(padded3.hex())

3a5e290505050505
3a5e29f09f9094f09fa59ae29d930202
3a5e29f09f9094f00808080808080808


### 3.2.b. Fonction d'unpadding

**TODO**  
Oter le padding des exemples précédents.

In [8]:
unpadder = pad.unpadder()
unpadded1 = unpadder.update(padded1) + unpadder.finalize()
print(unpadded1.hex())
unpadder = pad.unpadder()
unpadded2 = unpadder.update(padded2) + unpadder.finalize()
print(unpadded2.hex())
unpadder = pad.unpadder()
unpadded3 = unpadder.update(padded3) + unpadder.finalize()
print(unpadded3.hex())

3a5e29
3a5e29f09f9094f09fa59ae29d93
3a5e29f09f9094f0


### 3.2.c. Application du padding

**TODO**  
Utiliser ce *padding* pour chiffrer et déchiffrer le texte précédent
```txt
Ces données ne font pas un multiple de 16 octets.
```

In [10]:
data = "Ces données ne font pas un multiple de 16 octets.".encode()
pad = padding.PKCS7(algorithms.AES.block_size) # 128 bits (16 octets)

print("Données :", data)

# Rembourrage + chiffrement (côté Alice, avant envoi)
padder = pad.padder()
padded_data = padder.update(data) + padder.finalize()
print("Données rembourrées : ", padded_data)

encrypt = cipher.encryptor()
cipher_data = encrypt.update(padded_data) + encrypt.finalize()
print("Données chiffrées et rembourrées", cipher_data)

# Déchiffrement + dérembourrage (côté Bob, après réception des données)
decrypt = cipher.decryptor()
deciph_padded_data = decrypt.update(cipher_data) + decrypt.finalize()
unpadder = pad.unpadder()
print("Données déchiffrées mais toujours rembourrées :", deciph_padded_data)
deciph_data = unpadder.update(deciph_padded_data) + unpadder.finalize()
print("Données déchiffrées et dérembourrées :", deciph_data)

Données : b'Ces donn\xc3\xa9es ne font pas un multiple de 16 octets.'
Données rembourrées :  b'Ces donn\xc3\xa9es ne font pas un multiple de 16 octets.\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e'
Données chiffrées et rembourrées b'm\x88\xcf>e\xc1\xcd\xd3<\xd7x$Q\x18\xac\xe66\x1ao(\xc0\x1c\x8e\xbepl\x92\xae7xg\\\xb8\x8a@\x87\xee\x8e\x90\xf7z\xfc=k\x1f\x92=\xea\r\x93\xaf\xa6Zhu}\x85\x05/\xf7K\xe0\xb5\xf4'
Données déchiffrées mais toujours rembourrées : b'Ces donn\xc3\xa9es ne font pas un multiple de 16 octets.\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e\x0e'
Données déchiffrées et dérembourrées : b'Ces donn\xc3\xa9es ne font pas un multiple de 16 octets.'


## 3.3. Données en Base64

L'avantage de cette écriture est qu'elle est plus compacte que l'écriture hexadécimale (4 caractères en Base64 pour 6 caractères hexadécimaux). Sa description est détaillée dans le wiki du cours.

C'est utilisé pour l'échange de clés (pour les humains) et les messages LoRaWAN de l'*Internet of Things*.

In [ ]:
from base64 import b64decode, b64encode

### 3.3.a. Encodage Base64

L'encodage se fait avec la fonction `b64encode` du module `base64`.

La sortie est une liste d'octets `bytes` qui correspond aux caractères ASCII de l'écriture en base64.

**TODO**  
On reprend l'[exemple de Wikipédia](https://en.wikipedia.org/wiki/Base64#Decoding_with_padding). Vérifier que l'encodage de `b"light work."` donne bien `b"bGlnaHQgd29yay4="`.

In [ ]:
# TODO

**TODO (théorie)**  
En quoi est-ce différent de la méthode `bytes.hex()` ?

> Il y a deux différences : 
> 1. Un caractère en Base64 encode 6 bits alors qu'un caractère hex en encode 4.
> 2. La fonction `b64encode` renvoie un objet `bytes` alors que `bytes.hex()` renvoie un `str`.

### 3.3.b. Décodage Base64

Le décodage se fait avec `b64decode`. Un intérêt de cette fonction est qu'elle accepte un `bytes` ou un `str`.

**TODO**  
Décoder `Sm95IGFzIGFuIEFjdCBvZiBSZXNpc3RhbmNlLg==`. On devrait trouver le nom d'un album du groupe IDLES.

In [ ]:
# TODO

### 3.3.c. Transfert de données chiffrées

**TODO**  
Le message ci-dessous (en base64) a été chiffré avec la clé `ALL THEM WITCHES` :
```txt
Xp24aQbySTXWvWSx5Iej10KvplSA1wJQ7WIsGKNLJUsXDbtDBSK/RxI75m+MEI+F
7JP69mDI7pP8K4vAA+Sjf7iEK4XBFSOTZEdSJF811Gq8Gso9bnU5gDP+8Dx2aORf
```
Quelle est la chanson qui correspond à ces paroles ?

In [ ]:
data = b64decode(
    """
Xp24aQbySTXWvWSx5Iej10KvplSA1wJQ7WIsGKNLJUsXDbtDBSK/RxI75m+MEI+F
7JP69mDI7pP8K4vAA+Sjf7iEK4XBFSOTZEdSJF811Gq8Gso9bnU5gDP+8Dx2aORf
"""
)

## Pour aller plus loin

### Détecter le mode ECB

ATTENTION le mode de chiffrement ECB qu'on utilise ici est [vraiment pourri](https://crypto.stackexchange.com/questions/20941/why-shouldnt-i-use-ecb-encryption) ! Le seul cas d'usage devrait être dans le chiffrement d'un tout petit message ([26 octets](https://www.thethingsnetwork.org/docs/lorawan/end-device-activation/#step-4-1)) dans le protocole LoRaWAN pour l'*Internet of Things*.

Ce mode de chiffrement est plein de failles. S'il est utilisé en oracle, on peut facilement déchiffrer des données et faire des injections de code (e.g. dans des cookies). Il s'agit des défis [12](https://cryptopals.com/sets/2/challenges/12), [13](https://cryptopals.com/sets/2/challenges/13) et [14](https://cryptopals.com/sets/2/challenges/14) de Cryptopals.

En plus, il est très facile de détecter si ce mode de chiffrement est utilisé, vu que tout bloc. Si le message initial est très structuré, certains blocs vont se répéter et cette répétition va se retrouver dans le chiffrement. Un bon chiffrement devrait générer des blocs qui paraissent aléatoires les uns par rapport aux autres, et donc la probabilité d'une répétition serait $\sim 2^{-128}$. Donc si un bloc se répète, on peut être sûr que le mode ECB a été utilisé.

**TODO**  
Dans le fichier `detect.txt` se trouvent les hex de messages chiffrés. Seul l'un d'eux est chiffré en mode ECB. Lequel ?

> ligne 132

(il s'agit du [défi 8](https://cryptopals.com/sets/1/challenges/8) de Cryptopals)

In [ ]:
def detect_ecb(data: bytes, block_size: int = 16) -> bool:
    """
    Detect if the given data was encrypted using ECB mode.

    Args:
        data (bytes): The input data to analyze.
        block_size (int): Block size in bytes. Default is 16 bytes (128 bits).
    Returns:
        bool: True if ECB mode is detected, False otherwise.
    """
    # pass  # TODO

In [ ]:
with open("detect.txt", "r") as f:
    hexes = f.readlines()

#TODO